<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/TextFileToChromaDbAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

In [ ]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel

openai_api_key = userdata.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

In [ ]:
class QuoteLookupArgs(BaseModel):
  query: str
  # top_n: int

In [ ]:
QuoteLookupArgs.model_json_schema()

{'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'QuoteLookupArgs',
 'type': 'object'}

In [ ]:
# описанието да бъде с императивен изказ - работи по-добре, отколкото "този тул може да бъде извика за ...."
TOOLS = [
    {
        "type": "function",
        "name": "quote_lookup",
        "description": "Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query",
        "parameters": {
            **QuoteLookupArgs.model_json_schema(),
            "additionalProperties": False
        },
        "strict": True
    }
]

In [ ]:
TOOLS

[{'type': 'function',
  'name': 'quote_lookup',
  'description': 'Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query',
  'parameters': {'properties': {'query': {'title': 'Query', 'type': 'string'}},
   'required': ['query'],
   'title': 'QuoteLookupArgs',
   'type': 'object',
   'additionalProperties': False},
  'strict': True}]

In [ ]:
temp_response = openai_client.responses.create(
    model="gpt-5-mini",
    input=[
        {"role": "system", "content": "You are a helpful assitant for book analysis and quote finding. The user will enter a given topic and you need to use the \"quote_lookup\" tool to fing relevant quotes from the internal collection of books. Use it multiple times with different (similar) variations of the original user query. "},
        {"role": "user", "content": "Намери ми релевантни цитати, които са вързани със съотвените книги."}
    ],
    tools=TOOLS
)
